In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Build Results Fact
# MAGIC
# MAGIC 1. Read silver `results` table
# MAGIC 1. Read silver `sprints` table
# MAGIC 1. Add new column `session_type` with values `RACE` or `SPRINT`
# MAGIC 1. UNION `results` and `sprints`
# MAGIC 1. Derive additional columns
# MAGIC     - is_win -> Indicates that the driver own the race
# MAGIC     - is_podium -> Indicates that the driver scored a podium result (1, 2, 3)
# MAGIC     - has_points -> Indicates that the driver has scored points
# MAGIC 1. Write the transformed data to gold `fact_session_results` table
# MAGIC

In [0]:
%run ../00-common/01.environment_config

In [0]:
target_table = f"{catalog_name}.{gold_schema}.fact_session_results"

In [0]:
from pyspark.sql import functions as F

In [0]:
results_df = ( spark.table(f"{catalog_name}.{silver_schema}.results")
              .withColumn("session_type",F.lit("RACE"))
              .drop("race_name","race_date","ingestion_timestamp","source_file") )
results_df.display()


In [0]:
sprints_df = ( spark.table(f"{catalog_name}.{silver_schema}.sprints")
.withColumn("session_type",F.lit("SPRINT"))
.drop("race_name", "race_date", "ingestion_timestamp", "source_file") )
sprints_df.display()


In [0]:
sprints_df = sprints_df.withColumnRenamed("finish_position","final_position")


In [0]:
results_sprints_df = results_df.unionByName(sprints_df)

In [0]:
fact_session_results_df = (
    results_sprints_df
    .withColumn("is_win",F.col("final_position")==1)
    .withColumn("is_podium",F.col("final_position").between(1,3))
    .withColumn("has_points",F.col("points")>0)
)
fact_session_results_df.display(10)

In [0]:
fact_session_results_df.filter("season = 2025").display()

In [0]:
(
    fact_session_results_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
)

In [0]:
display(spark.table(target_table).filter("session_type != 'RACE'"))